In [1]:

# imports
import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import auc

import utils as ut

In [10]:
fs = 8000  # sampling frequency
f0 = 640  # signal frequency
duration = 60  # seconds
# snr_values = [-2, -4, -6, -8]
snr = -10
num_iterations = 1
tolerance = 5  # Hz

nperseg = 1000
nfft = None # 2*nperseg
noverlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 9
mode = "wasserstein"
quantization_bins = [2, 3, 5, 10, 20]

threshold = 0.02

In [11]:
raw_signal = ut.simulate_raw_signal(f0, fs, duration)
rx = ut.add_noise_to_signal(raw_signal, snr_db=snr, fs=fs, signal_bw=1, noise_type='white')

In [22]:
results = dict()
for qi, q in enumerate(quantization_bins):
    detector = ut.S2GDetector(fs=fs, nperseg=nperseg, overlap=noverlap, nfft=nfft, window=window, dc=dc, crop_freq=crop_freq, quantization_levels=q, mode=mode)
    is_detected, detections, (K, F) = detector.detect(rx, threshold)
    pd = 1 if np.any((F[detections] - f0) < tolerance) else 0
    fa = 1 if np.any((F[detections] - f0) > tolerance) else 0
    _res = dict()
    _res['is_detected'] = is_detected
    _res['detections'] = detections
    _res['K'] = K
    _res['F'] = F
    _res['pd'] = pd
    _res['fa'] = fa
    results[q] = _res

In [23]:
[results[q]['pd'] for q in quantization_bins], [results[q]['fa'] for q in quantization_bins]

([0, 1, 1, 1, 1], [0, 1, 0, 1, 1])

In [24]:
fig = make_subplots(rows=len(quantization_bins), cols=1)
for qi, q in enumerate(quantization_bins):
    fig.add_trace(go.Scatter(x=results[q]['F'], y=results[q]['K'], mode='lines', name=f'Q={q}'), row=qi+1, col=1)

    fig.add_trace(go.Scatter(
        x=results[q]['F'],
        y=threshold*np.ones(len(results[q]['F'])),
        mode='lines',
        name='threshold',
        line=dict(color='red'),
        showlegend=False
    ), row=qi+1, col=1)

fig.update_layout(height=len(quantization_bins)*100, width=1000)
fig.show()